# QA Benchmark — simple_questions_39_final_2025

Тест QA-пайплайна на датасете из 38 вопросов.  
Метрики: accuracy overall, по типу вопроса, по источнику документа, confidence calibration.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv('../.env')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('Setup OK')

## 1. Загрузка QA-движка

In [ ]:
from src.bot.engine_loader import load_engine

engine, _, _ = load_engine()
print('Engine loaded:', type(engine).__name__)

## 2. Загрузка датасета

In [ ]:
EXCEL_PATH = '../simple_questions_39_final_2025.xlsx'

df = pd.read_excel(EXCEL_PATH)
df.columns = ['question', 'expected', 'source_file', 'context_location', 'context', 'bot_answer_old', 'correct_old']
df = df[['question', 'expected', 'source_file', 'correct_old']].dropna(subset=['question', 'expected'])
df = df.reset_index(drop=True)

print(f'Loaded {len(df)} questions')
df[['question', 'expected', 'source_file']].head(5)

## 3. Запуск бенчмарка

In [ ]:
from tqdm.notebook import tqdm

TOP_K = 12
results = []

for _, row in tqdm(df.iterrows(), total=len(df), desc='Running QA'):
    q = str(row['question'])
    try:
        answer, ranked = engine.ask_full(q, top_k=TOP_K)
        confidence = ranked[0]['confidence'] if ranked else 0.0
        top3_facts = [r['text'] for r in ranked[:3] if r.get('_used_by_llm')]
        if not top3_facts:
            top3_facts = [r['text'] for r in ranked[:3]]
    except Exception as e:
        answer = f'ERROR: {e}'
        confidence = 0.0
        top3_facts = []

    results.append({
        'question': q,
        'expected': str(row['expected']),
        'bot_answer': str(answer),
        'confidence': round(confidence, 3),
        'source_file': str(row['source_file']),
        'correct_old': str(row.get('correct_old', '')),
        'top3_facts': ' | '.join(top3_facts),
    })

res_df = pd.DataFrame(results)
print(f'Done. {len(res_df)} results')

## 4. Разметка результатов

Ручная разметка: `correct` = 1 (верно), 0.5 (частично), 0 (неверно).

In [ ]:
def auto_label(row):
    """Автоматическая эвристика: точное совпадение подстроки."""
    exp = str(row['expected']).strip().lower()
    got = str(row['bot_answer']).strip().lower()
    if got in ('unknown', 'null', 'none', ''):
        return 0
    if exp in got or got in exp:
        return 1
    # Числовое совпадение: убрать пробелы и сравнить
    exp_num = exp.replace(' ', '').replace(',', '.')
    got_num = got.replace(' ', '').replace(',', '.')
    if exp_num and exp_num in got_num:
        return 1
    return 0

res_df['auto_correct'] = res_df.apply(auto_label, axis=1)
res_df['is_unknown'] = res_df['bot_answer'].str.lower().isin(['unknown', 'null', 'none', ''])

# Показать таблицу
pd.set_option('display.max_colwidth', 60)
res_df[['question', 'expected', 'bot_answer', 'auto_correct', 'confidence']].style.apply(
    lambda row: ['background: #d4edda' if row['auto_correct'] == 1
                 else 'background: #f8d7da' if row['auto_correct'] == 0
                 else '' for _ in row], axis=1
)

## 5. Общие метрики

In [ ]:
n = len(res_df)
n_correct = res_df['auto_correct'].sum()
n_unknown = res_df['is_unknown'].sum()
n_wrong = n - n_correct - n_unknown

print(f'Total questions  : {n}')
print(f'Correct          : {n_correct} ({n_correct/n*100:.1f}%)')
print(f'Unknown (no ans) : {n_unknown} ({n_unknown/n*100:.1f}%)')
print(f'Wrong answer     : {n_wrong} ({n_wrong/n*100:.1f}%)')
print()
print(f'Baseline (old)   : 34.2% correct')
delta = n_correct/n*100 - 34.2
print(f'Delta vs baseline: {delta:+.1f}%')

## 6. Разбивка по документам-источникам

In [ ]:
by_source = res_df.groupby('source_file').agg(
    total=('auto_correct', 'count'),
    correct=('auto_correct', 'sum'),
    unknown=('is_unknown', 'sum'),
    avg_conf=('confidence', 'mean'),
).reset_index()
by_source['accuracy'] = (by_source['correct'] / by_source['total'] * 100).round(1)
by_source = by_source.sort_values('accuracy')

# Short names for display
by_source['doc'] = by_source['source_file'].str[:50]

fig, ax = plt.subplots(figsize=(12, 7))
colors = ['#d4edda' if a >= 50 else '#f8d7da' for a in by_source['accuracy']]
ax.barh(by_source['doc'], by_source['accuracy'], color=colors)
ax.axvline(34.2, color='gray', linestyle='--', label='Baseline 34.2%')
ax.set_xlabel('Accuracy %')
ax.set_title('Accuracy by source document')
ax.legend()
plt.tight_layout()
plt.show()

by_source[['doc', 'total', 'correct', 'unknown', 'accuracy', 'avg_conf']]

## 7. Calibration: confidence vs correctness

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter
ax = axes[0]
colors_scatter = ['#2ca02c' if c == 1 else '#d62728' for c in res_df['auto_correct']]
ax.scatter(res_df.index, res_df['confidence'], c=colors_scatter, alpha=0.7, s=60)
ax.set_xlabel('Question index')
ax.set_ylabel('Confidence')
ax.set_title('Confidence per question (green=correct, red=wrong)')
ax.set_ylim(0, 1.05)

# Box plot
ax2 = axes[1]
conf_correct = res_df[res_df['auto_correct'] == 1]['confidence']
conf_wrong   = res_df[res_df['auto_correct'] == 0]['confidence']
ax2.boxplot([conf_correct, conf_wrong], labels=['Correct', 'Wrong'])
ax2.set_ylabel('Confidence')
ax2.set_title('Confidence distribution: correct vs wrong')

plt.tight_layout()
plt.show()

print(f"Mean confidence — Correct: {conf_correct.mean():.3f}  |  Wrong: {conf_wrong.mean():.3f}")
if conf_wrong.mean() > conf_correct.mean():
    print('⚠ Miscalibration: model is MORE confident on wrong answers')
else:
    print('✓ Calibration OK: model is more confident on correct answers')

## 8. Разбивка ошибок по типу

In [ ]:
print('=== Unknown answers (retrieval failures) ===')
unknown_df = res_df[res_df['is_unknown']]
for _, r in unknown_df.iterrows():
    print(f"  Q: {r['question'][:80]}")
    print(f"     Expected: {r['expected']}  |  Source: {r['source_file'][:40]}")
    print()

print('=== Wrong answers (extraction errors) ===')
wrong_df = res_df[(res_df['auto_correct'] == 0) & (~res_df['is_unknown'])]
for _, r in wrong_df.iterrows():
    print(f"  Q: {r['question'][:80]}")
    print(f"     Expected: {r['expected']!r}  |  Got: {r['bot_answer']!r}  [conf={r['confidence']}]")
    print(f"     Top facts: {r['top3_facts'][:120]}")
    print()

## 9. Сохранение результатов

In [ ]:
out_path = '../benchmark_results.xlsx'
res_df.to_excel(out_path, index=False)
print(f'Results saved to {out_path}')